In [1]:
import pandas as pd

# Carrega a base integrada de migração e emprego
indicadores_integrados = pd.read_parquet(
    "../data/processed/indicadores_integrados_municipais.parquet"
)

# Carrega os indicadores municipais de educação técnica
educacao_municipal = pd.read_parquet(
    "../data/processed/educacao/indicadores_educacao_municipal_2025.parquet"
)

# Integra as três dimensões pela chave municipal do IBGE
indicadores_completos = indicadores_integrados.merge(
    educacao_municipal.drop(columns=["municipio", "uf"], errors="ignore"),
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_integracao_educacao"
)

# Confere dimensões e cobertura entre as bases
print("Migração + emprego:", indicadores_integrados.shape)
print("Educação:", educacao_municipal.shape)
print("Base completa:", indicadores_completos.shape)

print("\nCobertura da integração:")
print(indicadores_completos["origem_integracao_educacao"].value_counts())

# Verifica se existem códigos IBGE duplicados
print("\nCódigos IBGE duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

Migração + emprego: (1228, 12)
Educação: (3385, 10)
Base completa: (3689, 20)

Cobertura da integração:
origem_integracao_educacao
right_only    2461
both           924
left_only      304
Name: count, dtype: int64

Códigos IBGE duplicados: 0


In [2]:
# Carrega a referência oficial dos municípios
municipios_ibge = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Adiciona os nomes e UFs de todos os municípios
indicadores_completos = indicadores_completos.drop(
    columns=["municipio", "uf"], errors="ignore"
).merge(
    municipios_ibge[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left",
    validate="many_to_one"
)

# Verifica identificações geográficas ausentes
print("Municípios sem nome:", indicadores_completos["municipio"].isna().sum())
print("Municípios sem UF:", indicadores_completos["uf"].isna().sum())

# Verifica os valores ausentes em todas as colunas
display(
    indicadores_completos.isna().sum()
    .sort_values(ascending=False)
    .to_frame("valores_ausentes")
)

Municípios sem nome: 0
Municípios sem UF: 0


,valores_ausentes
variacao_%,2942
2024,2755
2025,2755
variacao_absoluta,2755
2026_parcial,2755
saldo,2635
admissoes,2635
desligamentos,2635
_merge,2461
percentual_ead,345


In [3]:
# Identifica os municípios presentes nas fontes originais
indicadores_completos["tem_migracao"] = (
    indicadores_completos["2025"].notna()
)

indicadores_completos["tem_emprego"] = (
    indicadores_completos["admissoes"].notna()
)

indicadores_completos["tem_educacao"] = (
    indicadores_completos["ofertas_cursos"].notna()
)

# Conta quantas dimensões possuem dados em cada município
indicadores_completos["fontes_disponiveis"] = (
    indicadores_completos[
        ["tem_migracao", "tem_emprego", "tem_educacao"]
    ].sum(axis=1)
)

# Resume a cobertura das três dimensões
display(
    indicadores_completos["fontes_disponiveis"]
    .value_counts()
    .sort_index()
    .to_frame("municipios")
)

# Identifica quantos municípios possuem dados nas três dimensões
print(
    "Municípios com três dimensões:",
    (indicadores_completos["fontes_disponiveis"] == 3).sum()
)


,municipios
fontes_disponiveis,
1,2628
2,438
3,623


Municípios com três dimensões: 623


In [4]:
# Agrupa municípios pelas combinações de fontes disponíveis
cobertura_fontes = (
    indicadores_completos.groupby(
        ["tem_migracao", "tem_emprego", "tem_educacao"]
    )
    .size()
    .reset_index(name="municipios")
)

# Ordena as combinações pelo número de municípios
cobertura_fontes = cobertura_fontes.sort_values(
    "municipios", ascending=False
)

# Exibe a distribuição
display(cobertura_fontes)

,tem_migracao,tem_emprego,tem_educacao,municipios
0,False,False,True,2461
6,True,True,True,623
2,False,True,True,180
5,True,True,False,137
4,True,False,True,121
1,False,True,False,114
3,True,False,False,53


In [5]:
# Seleciona municípios com registros de migração e emprego
municipios_priorizacao = indicadores_completos[
    indicadores_completos["tem_migracao"]
    & indicadores_completos["tem_emprego"]
].copy()

# Mantém informações territoriais e indicadores relevantes
colunas_priorizacao = [
    "codigo_ibge", "municipio", "uf",
    "2024", "2025", "variacao_absoluta", "variacao_%",
    "admissoes", "desligamentos", "saldo",
    "instituicoes_tecnicas", "cursos_distintos",
    "matriculas_tecnicas", "percentual_ead"
]

municipios_priorizacao = municipios_priorizacao[
    colunas_priorizacao
]

# Confere tamanho e cobertura educacional da base principal
print("Municípios elegíveis:", len(municipios_priorizacao))
print(
    "Com dados educacionais:",
    municipios_priorizacao["instituicoes_tecnicas"].notna().sum()
)

# Visualiza os primeiros registros
display(municipios_priorizacao.head())

Municípios elegíveis: 760
Com dados educacionais: 623


,codigo_ibge,municipio,uf,2024,2025,variacao_absoluta,variacao_%,admissoes,desligamentos,saldo,instituicoes_tecnicas,cursos_distintos,matriculas_tecnicas,percentual_ead
1,1100023,ARIQUEMES,RO,0.0,17.0,17.0,NaN,34.0,18.0,16.0,4.0,13.0,1185.0,30.23
2,1100049,CACOAL,RO,94.0,89.0,-5.0,-5.32,160.0,141.0,19.0,7.0,10.0,1389.0,34.09
3,1100056,CEREJEIRAS,RO,10.0,24.0,14.0,140.00,10.0,0.0,10.0,1.0,1.0,38.0,0.00
7,1100098,ESPIGÃO D'OESTE,RO,19.0,14.0,-5.0,-26.32,40.0,45.0,-5.0,NaN,NaN,NaN,NaN
8,1100106,GUAJARÁ-MIRIM,RO,143.0,130.0,-13.0,-9.09,26.0,16.0,10.0,1.0,5.0,534.0,0.00


In [6]:
# Carrega os indicadores municipais do ensino superior
superior_municipal = pd.read_parquet(
    "../data/processed/educacao/indicadores_ensino_superior_municipal_2024.parquet"
)

# Seleciona apenas a chave municipal e os indicadores educacionais
colunas_superior = [
    "codigo_ibge",
    "cursos_presenciais",
    "ies_presenciais",
    "matriculas_presenciais",
    "matriculas_estrangeiros_presencial",
    "matriculas_ead",
    "ingressantes_ead",
    "matriculas_estrangeiros_ead"
]

# Integra o ensino superior, preservando municípios de todas as fontes
indicadores_completos = indicadores_completos.merge(
    superior_municipal[colunas_superior],
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_integracao_superior"
)

# Verifica a integração
print("Dimensões:", indicadores_completos.shape)
print("\nCobertura:")
print(indicadores_completos["origem_integracao_superior"].value_counts())

print("\nCódigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

print("Municípios sem nome:",
      indicadores_completos["municipio"].isna().sum())

Dimensões: (4441, 32)

Cobertura:
origem_integracao_superior
both          2799
left_only      890
right_only     752
Name: count, dtype: int64

Códigos duplicados: 0
Municípios sem nome: 752


In [7]:
# Carrega a referência oficial dos municípios
municipios_ibge = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Adiciona os nomes e UFs oficiais como colunas auxiliares
indicadores_completos = indicadores_completos.merge(
    municipios_ibge[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left",
    suffixes=("", "_ibge"),
    validate="one_to_one"
)

# Preenche apenas os nomes e UFs que estavam ausentes
indicadores_completos["municipio"] = (
    indicadores_completos["municipio"]
    .fillna(indicadores_completos["municipio_ibge"])
)

indicadores_completos["uf"] = (
    indicadores_completos["uf"]
    .fillna(indicadores_completos["uf_ibge"])
)

# Remove as colunas auxiliares
indicadores_completos = indicadores_completos.drop(
    columns=["municipio_ibge", "uf_ibge"]
)

# Valida a identificação territorial
print("Municípios:", len(indicadores_completos))
print("Sem nome:", indicadores_completos["municipio"].isna().sum())
print("Sem UF:", indicadores_completos["uf"].isna().sum())
print("Códigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

Municípios: 4441
Sem nome: 0
Sem UF: 0
Códigos duplicados: 0


In [8]:
# Atualiza os indicadores de presença das fontes
indicadores_completos["tem_migracao"] = indicadores_completos["2025"].notna()
indicadores_completos["tem_emprego"] = indicadores_completos["admissoes"].notna()
indicadores_completos["tem_educacao"] = indicadores_completos["ofertas_cursos"].notna()

# Identifica municípios com registros de ensino superior presencial ou EAD
indicadores_completos["tem_superior"] = indicadores_completos[
    ["matriculas_presenciais", "matriculas_ead"]
].notna().any(axis=1)

# Recalcula a quantidade de dimensões disponíveis
colunas_cobertura = [
    "tem_migracao",
    "tem_emprego",
    "tem_educacao",
    "tem_superior"
]

indicadores_completos["fontes_disponiveis"] = (
    indicadores_completos[colunas_cobertura].sum(axis=1)
)

# Exibe a distribuição de cobertura
display(
    indicadores_completos["fontes_disponiveis"]
    .value_counts()
    .sort_index()
    .to_frame("municipios")
)

# Confere a cobertura individual de cada fonte
display(
    indicadores_completos[colunas_cobertura]
    .sum()
    .to_frame("municipios")
)

,municipios
fontes_disponiveis,
1,1536
2,1919
3,394
4,592


,municipios
tem_migracao,934
tem_emprego,1054
tem_educacao,3385
tem_superior,3551


In [9]:
# Remove colunas auxiliares utilizadas durante as integrações
indicadores_completos = indicadores_completos.drop(
    columns=[
        "_merge",
        "origem_integracao_educacao",
        "origem_integracao_superior"
    ],
    errors="ignore"
)

# Confere os totais e a integridade da base
print("Municípios:", len(indicadores_completos))
print("Códigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

print("Registros migratórios 2025:",
      indicadores_completos["2025"].sum())

print("Admissões CAGED 2025:",
      indicadores_completos["admissoes"].sum())

print("Matrículas técnicas 2025:",
      indicadores_completos["matriculas_tecnicas"].sum())

print("Matrículas superiores presenciais 2024:",
      indicadores_completos["matriculas_presenciais"].sum())

print("Matrículas superiores EAD 2024:",
      indicadores_completos["matriculas_ead"].sum())

Municípios: 4441
Códigos duplicados: 0
Registros migratórios 2025: 236777.0
Admissões CAGED 2025: 393415.0
Matrículas técnicas 2025: 2490145.0
Matrículas superiores presenciais 2024: 5037875.0
Matrículas superiores EAD 2024: 5186811.0


In [10]:
# Define o caminho da base municipal consolidada
arquivo_saida = "../data/processed/indicadores_completos_municipais.parquet"

# Salva os indicadores integrados sem incluir o índice do DataFrame
indicadores_completos.to_parquet(
    arquivo_saida,
    index=False
)

# Confirma a exportação e suas dimensões
print("Arquivo salvo:", arquivo_saida)
print("Dimensões:", indicadores_completos.shape)

Arquivo salvo: ../data/processed/indicadores_completos_municipais.parquet
Dimensões: (4441, 30)
